# M02B · Parsing & Manipulating FASTA & FASTQ Files by Hand
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2024-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M02_File_Handling/solutions/M02B_FASTA_by_Hand_Solutions.ipynb)

⏱ ~90 min · Difficulty: ★★☆ · Prerequisites: M02A (Text Files, Context Managers, Line Streaming)

---


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Explain** the technical specification and biological purpose of FASTA and FASTQ format standards in genomics.
2. **Implement** robust multi-line FASTA parsers using stateful accumulator loops without relying on third-party libraries.
3. **Construct** standard-compliant FASTA output files with fixed 60-character sequence line wrapping.
4. **Filter and split** multi-record genomic files based on sequence length, GC content, and custom biological criteria.
5. **Decode** FASTQ quality strings using Phred+33 ASCII offset mapping and compute per-read mean quality scores and error probabilities.
6. **Identify** edge cases (wrapped lines, trailing records, IUPAC degenerate codes) that necessitate specialized parsers in production pipelines.

---


## 2. Prerequisites

Before starting this notebook, ensure you understand:
- Reading and writing files in Python using context managers (`with open(...) as f:`).
- Line stripping (`.strip()`) and string manipulation (`.upper()`, `.startswith()`, `.split()`).
- Basic DNA biology: nucleotide alphabets (A, C, G, T, N), GC content calculations, and reverse complements.
- Completed **M02A (Text Files in Python)**.

---


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

# Detect whether we are running in Google Colab or locally
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root
    ROOT = Path.cwd().resolve()
    while not (ROOT / "data").exists() and ROOT.parent != ROOT:
        ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M02B"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python interpreter : {sys.version.split()[0]}")
print(f"Repository ROOT    : {ROOT.resolve()}")
print(f"DATA_DIR exists    : {DATA_DIR.exists()} ({DATA_DIR})")
print(f"OUT_DIR ready      : {OUT_DIR.exists()} ({OUT_DIR})")


## 4. Concept Sections

### 4.1 Concept 1: The FASTA Specification & Multi-Line Sequence Wrapping

The **FASTA format** is the universal standard for representing nucleotide (DNA/RNA) and peptide (protein) sequences in computational biology. Originally introduced in 1985 by William Pearson for the FASTP/FASTA sequence alignment program, it remains the standard across NCBI, EMBL-EBI, Ensembl, and UniProt.

#### Anatomy of a FASTA Record
```text
>seq_id optional description text goes here
ATGCGTACGTTAGCTAGCTAGCTAGCTAGCTAGCTAGCTA
CGTACGTTAGCTAGCTAGCTA
```
1. **Header Line**:
   - Always begins with a greater-than sign (`>`).
   - The characters immediately following `>` up to the first whitespace represent the unique **Sequence Identifier** (e.g., `NM_000546.6` or `sample_01`).
   - Any text following the first whitespace is the optional **Description / Metadata** (e.g., `Homo sapiens tumor protein p53`).
2. **Sequence Lines**:
   - Represent the biological sequence (nucleotides or amino acids).
   - **Crucial Rule:** In standard FASTA files, the sequence is almost always **wrapped across multiple lines** (typically 60, 70, or 80 characters per line) for human readability in terminal pagers.
   - Whitespace and numbers are not permitted within sequence lines.
3. **Multi-FASTA Files**:
   - Contain multiple records concatenated together in a single file. Each record begins with its own `>` header line.


In [ ]:
# Worked Example 4.1: Inspecting a real multi-FASTA file
fasta_path = DATA_DIR / "sequences" / "multi_sample.fasta"

# Inspect the first 12 lines of the file
with open(fasta_path, "r", encoding="utf-8") as f:
    for i in range(12):
        print(f"Line {i+1:2d}: {repr(f.readline())}")


#### ✏️ Try it 4.1: Count FASTA Headers
Count how many headers (lines starting with `>`) exist in `multi_sample.fasta`.


In [ ]:
# Try it 4.1: Count headers in multi_sample.fasta
header_count = 0
with open(DATA_DIR / "sequences" / "multi_sample.fasta", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith(">"):
            header_count += 1

print(f"Found {header_count} records.")
assert header_count == 15, f"Expected 15 headers, found {header_count}"
print("Try it 4.1 passed ✔")


#### ⚠️ Common Mistakes with FASTA Format
- **Assuming one sequence line per header**: Beginners often write code assuming `header = f.readline()` and `seq = f.readline()`. This completely breaks on wrapped multi-line FASTA files!
- **Splitting header without stripping `>`**: Writing `header.split()[0]` includes the `>` sign in the ID (e.g. `">sample_01"`). You must strip the leading `>`: `header[1:].split()[0]`.
- **Assuming uppercase only**: FASTA files often use lowercase letters (e.g. `atgc`) to denote soft-masked repeat regions. Always call `.upper()` before analyzing composition.


### 4.2 Concept 2: Hand-Crafted FASTA Parser (The Accumulator Pattern)

Because sequence lines are wrapped over multiple lines, parsing a FASTA file requires a stateful loop known as the **accumulator pattern**:

```text
Initialize: current_id = None, current_seq_chunks = [], records = {}

For each line in file:
    If line starts with '>':
        If current_id is not None:
            Save previous record: records[current_id] = "".join(current_seq_chunks)
        Parse new ID from line
        Reset current_seq_chunks = []
    Else:
        Append cleaned sequence line to current_seq_chunks

After loop finishes:
    If current_id is not None:
        Save the LAST record: records[current_id] = "".join(current_seq_chunks)
```

#### The Notorious "Last Record" Bug
Notice the critical step **after the loop finishes**. In a file with 15 records, the 15th record does NOT have a subsequent `>` header line to trigger saving! If you omit the post-loop save, your parser will silently discard the final record in every file it processes.


In [ ]:
# Worked Example 4.2: Robust FASTA parser by hand
def read_fasta_by_hand(filepath: Path) -> dict[str, str]:
    """Parse a FASTA file into a dictionary of {seq_id: full_sequence}."""
    records = {}
    current_id = None
    chunks = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            cleaned = line.strip()
            if not cleaned:
                continue  # Skip blank lines
            
            if cleaned.startswith(">"):
                # Save previous sequence if one was being accumulated
                if current_id is not None:
                    records[current_id] = "".join(chunks)
                
                # Extract sequence ID (first word after '>')
                header_text = cleaned[1:].strip()
                current_id = header_text.split()[0]
                chunks = []
            else:
                chunks.append(cleaned.upper())

        # CRITICAL: Save the very last record after the file iteration ends
        if current_id is not None:
            records[current_id] = "".join(chunks)

    return records

# Test on multi_sample.fasta
parsed_dict = read_fasta_by_hand(DATA_DIR / "sequences" / "multi_sample.fasta")
print(f"Successfully parsed {len(parsed_dict)} records.")
first_key = list(parsed_dict.keys())[0]
print(f"First record [{first_key}]: length {len(parsed_dict[first_key])} bp")
print(f"First 50 bp: {parsed_dict[first_key][:50]}...")


#### ✏️ Try it 4.2: Verify Record Integrity
Using `read_fasta_by_hand`, verify that `sample_01` has length 407 bp and `sample_15` is present in the dictionary.


In [ ]:
# Try it 4.2: Verify record parsing
records = read_fasta_by_hand(DATA_DIR / "sequences" / "multi_sample.fasta")
assert len(records) == 15, f"Expected 15 records, got {len(records)}"
assert "sample_01" in records, "sample_01 must be present"
assert len(records["sample_01"]) == 407, f"Expected 407 bp, got {len(records['sample_01'])}"
assert "sample_15" in records, "sample_15 (the last record) must be preserved!"
print("Try it 4.2 passed ✔")


#### ⚠️ Common Mistakes in FASTA Parsers
- **Omitting the post-loop save**: Silently discarding the last sequence in every FASTA file.
- **Using string concatenation `seq += line` inside the loop**: In Python, strings are immutable. Repeated `seq += line` creates a brand-new string copy on every line, degrading performance to quadratic $O(N^2)$ time. Appending to a list `chunks.append(line)` and joining once with `"".join(chunks)` is linear $O(N)$ and orders of magnitude faster.
- **Not stripping whitespace**: Failing to strip trailing `\r` or `\n` embeds invisible newline characters inside your sequence strings, corrupting downstream translation and alignment.


### 4.3 Concept 3: Writing Standard Wrapped FASTA Files

When writing sequences to FASTA format:
1. Always start with a `>` header line.
2. Standard practice is to wrap the sequence at **60 or 70 characters per line**.
3. Slicing sequences into fixed-width lines is cleanly accomplished using a `range(0, len(seq), wrap)` stride:
   ```python
   for i in range(0, len(seq), 60):
       chunk = seq[i : i + 60]
       f.write(chunk + "\n")
   ```

#### Why Line Wrapping Matters
While some modern tools accept unwrapped sequences, legacy tools (e.g. NCBI BLAST+ command line, FASTA36, EMBOSS) have fixed buffer sizes for reading input lines. Unwrapped sequences spanning hundreds of thousands of base pairs on a single line can cause buffer overflow segmentation faults.


In [ ]:
# Worked Example 4.3: Writing wrapped FASTA records
def write_fasta_wrapped(records: dict[str, str], out_path: Path, wrap_width: int = 60) -> int:
    """Write a dictionary of {id: sequence} to a formatted FASTA file."""
    count = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for seq_id, seq in records.items():
            f.write(f">{seq_id}\n")
            for i in range(0, len(seq), wrap_width):
                f.write(seq[i : i + wrap_width] + "\n")
            count += 1
    return count

# Export first two records wrapped at 60 bp
subset = {k: parsed_dict[k] for k in list(parsed_dict.keys())[:2]}
test_out = OUT_DIR / "wrapped_demo.fasta"
write_fasta_wrapped(subset, test_out, wrap_width=60)

with open(test_out, "r", encoding="utf-8") as f:
    for line in f.readlines()[:6]:
        print(" ", repr(line.strip()))


#### ✏️ Try it 4.3: Slice Sequence into Wrapped Lines
Given a 130-nt synthetic sequence, slice it into chunks of width 60 and verify the line lengths.


In [ ]:
# Try it 4.3: Test line slicing
synthetic_seq = "A" * 60 + "C" * 60 + "G" * 10
chunks = [synthetic_seq[i : i + 60] for i in range(0, len(synthetic_seq), 60)]

assert len(chunks) == 3, f"Expected 3 chunks, got {len(chunks)}"
assert [len(c) for c in chunks] == [60, 60, 10]
print("Try it 4.3 passed ✔")


#### ⚠️ Common Mistakes when Writing FASTA
- **Writing one massive line**: Unwrapped chromosomes (e.g. 248 million bp for human chromosome 1) will crash standard text viewers and many CLI tools.
- **Missing newline between records**: Forgetting `\n` at the end of the last line of a sequence causes the subsequent record's `>` header to fuse with the sequence.


### 4.4 Concept 4: Filtering and Splitting Multi-FASTA Files

Common biological preprocessing workflows include:
1. **Length filtering**: Removing short reads (< 50 bp) or short contigs (< 500 bp) prior to assembly.
2. **GC content filtering**: Partitioning sequences by nucleotide composition (e.g., separating low-GC AT-rich regions from high-GC isochores).
3. **Splitting into individual files**: Distributing a dataset of 1,000 sequences across multiple CPU cores on an HPC cluster by saving one sequence per file.


In [ ]:
# Worked Example 4.4: Filtering sequences by GC content
def gc_content(seq: str) -> float:
    """Calculate GC percentage of a nucleotide sequence."""
    if not seq:
        return 0.0
    upper_seq = seq.upper()
    gc = upper_seq.count("G") + upper_seq.count("C")
    return round((gc / len(upper_seq)) * 100.0, 2)

# Filter records with GC >= 50%
high_gc_records = {
    seq_id: seq for seq_id, seq in parsed_dict.items()
    if gc_content(seq) >= 50.0
}

print(f"Total sequences       : {len(parsed_dict)}")
print(f"High-GC sequences (>=50%): {len(high_gc_records)}")
for seq_id, seq in high_gc_records.items():
    print(f"  {seq_id}: length={len(seq)} bp, GC={gc_content(seq)}%")


#### ✏️ Try it 4.4: Filter by Length
Filter `parsed_dict` to find all sequences longer than 300 bp and count them.


In [ ]:
# Try it 4.4: Length filter
long_seqs = {k: v for k, v in parsed_dict.items() if len(v) > 300}
print(f"Sequences > 300 bp: {len(long_seqs)}")
assert len(long_seqs) == 4, f"Expected 4 sequences > 300 bp, found {len(long_seqs)}"
print("Try it 4.4 passed ✔")


#### ⚠️ Common Mistakes in Filtering
- **Modifying dictionary while iterating**: Calling `del records[k]` inside `for k in records:` raises `RuntimeError: dictionary changed size during iteration`. Always construct a new dictionary or list comprehension.
- **Division by zero**: Calculating `gc_count / len(seq)` on an empty sequence without checking `if len(seq) == 0`.


### 4.5 Concept 5: The FASTQ Format & Phred Quality Scores

The **FASTQ format** extends FASTA to include per-base sequencing quality scores. It is the primary format emitted by high-throughput sequencing instruments (Illumina, PacBio, Oxford Nanopore).

#### The 4-Line Structure
Every FASTQ record consists of exactly **4 lines**:
```text
@READ_001 length=50
AGTATGAAGTTCAGGTTGTGACCAATGGACCGCAGGGGTGGATATCATCC
+
G=D?E@?HCIIEAG?AGIHGB=EE?>==FFCEE>A@HHHB@CGE???@FA
```
1. **Line 1 (`@`)**: Header beginning with `@`, containing the read ID and sequencer instrument metadata.
2. **Line 2**: Raw nucleotide sequence call (A, C, G, T, N).
3. **Line 3 (`+`)**: Separator line beginning with `+` (optionally repeats the read ID).
4. **Line 4**: ASCII quality score string. **Must have the exact same length as Line 2!**

#### Phred Quality Scores (Phred+33)
Each ASCII character in Line 4 encodes a numerical **Phred quality score ($Q$)**:
$$Q = \text{ord}(\text{char}) - 33$$
where $\text{ord}()$ is Python's built-in returning the integer ASCII code of a character.

The Phred score relates to the base-calling error probability ($P_{\text{error}}$):
$$P_{\text{error}} = 10^{-Q / 10}$$

| Phred Score ($Q$) | Error Probability ($P$) | Base Call Accuracy | Meaning in Genomics |
|---|---|---|---|
| **$Q10$** | $1 \text{ in } 10$ ($0.1$) | $90.0\%$ | Low quality; high error rate |
| **$Q20$** | $1 \text{ in } 100$ ($0.01$) | $99.0\%$ | Acceptable minimum threshold |
| **$Q30$** | $1 \text{ in } 1,000$ ($0.001$) | $99.9\%$ | **Standard benchmark for high-quality reads** |
| **$Q40$** | $1 \text{ in } 10,000$ ($0.0001$) | $99.99\%$ | Extremely high accuracy |


In [ ]:
# Worked Example 4.5: Parsing FASTQ and decoding Phred+33 scores
fastq_file = DATA_DIR / "sequences" / "reads_small.fastq"

def parse_fastq_hand(filepath: Path):
    """Parse FASTQ file yielding (read_id, seq, phred_scores)."""
    with open(filepath, "r", encoding="utf-8") as f:
        while True:
            line1 = f.readline()
            if not line1:
                break  # End of file
            line2 = f.readline()
            line3 = f.readline()
            line4 = f.readline()
            
            read_id = line1.strip()[1:].split()[0]
            seq = line2.strip().upper()
            qual_str = line4.strip()
            
            # Decode ASCII characters to Phred integers
            phred_scores = [ord(c) - 33 for c in qual_str]
            yield read_id, seq, phred_scores

# Inspect first read
for read_id, seq, scores in parse_fastq_hand(fastq_file):
    mean_q = sum(scores) / len(scores)
    print(f"Read: {read_id} | Length: {len(seq)} bp | Mean Q: {mean_q:.2f}")
    print(f"First 10 bases  : {seq[:10]}")
    print(f"First 10 scores : {scores[:10]}")
    break


#### ✏️ Try it 4.5: Decode Phred String
Decode the quality score string `"IIIEAG"` into integer Phred scores and calculate their mean.


In [ ]:
# Try it 4.5: Decode Phred string
sample_qual = "IIIEAG"
scores = [ord(c) - 33 for c in sample_qual]
mean_val = sum(scores) / len(scores)

print("Scores:", scores, "Mean:", mean_val)
assert scores == [40, 40, 40, 36, 32, 38], f"Got: {scores}"
assert round(mean_val, 2) == 37.67
print("Try it 4.5 passed ✔")


#### ⚠️ Common Mistakes with FASTQ
- **Assuming Phred+64 for modern data**: Older Solexa/Illumina 1.3–1.7 pipelines used Phred+64. Since Illumina 1.8+ and across all modern sequencers, Phred+33 is the universal standard.
- **Desynchronizing the 4-line cycle**: Blank lines inside FASTQ files throw off the 4-line cadence, causing line 4 (quality) to be parsed as line 1 (header).
- **Ignoring line length equality**: If `len(seq) != len(qual)`, the FASTQ file is truncated or corrupt. Always validate equality!


### 4.6 Concept 6: Pitfalls of Hand-Parsing & The Need for Biopython

Writing custom parsers gives deep insight into file structure, but production bioinformatics pipelines rely on established libraries like **Biopython** (`Bio.SeqIO`). Why?

1. **Complex Headers**: Real FASTA files have headers with pipes (`gi|12345|ref|NM_000546.6|`), key-value pairs, or multi-line metadata.
2. **Ambiguous & Degenerate Bases**: Biological sequences include IUPAC ambiguity codes (`R` = A/G, `Y` = C/T, `N` = any, `-` = gap). Hand-crafted parsers often fail to account for them.
3. **Massive File Sizes**: A single Illumina NovaSeq run generates hundreds of gigabytes of gzipped FASTQ (`.fastq.gz`). Hand-rolled parsers that load data into Python dictionaries exhaust RAM instantly.
4. **Format Diversity**: Converting between FASTA, FASTQ, GenBank, EMBL, and BAM requires hundreds of edge-case rules that `Bio.SeqIO` has solved and tested over 20+ years.


---
## 5. Practice Exercises

Apply your parsing and sequence manipulation skills to solve these exercises:
- **Level 1 (★)**: Direct header and sequence length extraction.
- **Level 2 (★★)**: Multi-FASTA filtering, splitting, and FASTQ quality parsing.
- **Level 3 (★★★)**: Chunked multi-file batch splitting.

---


### Ex 2B.1 (★) — Extract FASTA Headers and Descriptions

**Problem Statement:**  
Write a function `parse_fasta_headers(filepath: Path) -> dict[str, str]` that:
1. Reads a FASTA file.
2. Extracts every header line.
3. Maps the sequence ID (the first non-whitespace token after `>`) to the full description text (all text following the ID). If there is no description, map to `""`.
4. Returns the dictionary `{seq_id: description}`.

- **Input:** `filepath: Path`.
- **Output:** `dict[str, str]`.


In [ ]:
def parse_fasta_headers(filepath: Path) -> dict[str, str]:
    headers = {}
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            cleaned = line.strip()
            if cleaned.startswith(">"):
                # Remove leading '>'
                content = cleaned[1:].strip()
                parts = content.split(maxsplit=1)
                seq_id = parts[0]
                desc = parts[1] if len(parts) > 1 else ""
                headers[seq_id] = desc
    return headers


**Explanation:**
- `content.split(maxsplit=1)` splits the header string at most once, cleanly separating the sequence identifier from the entire remaining description text.
- If no description exists, `len(parts) == 1`, so we default to `""`.
- Only lines starting with `>` are processed, keeping the function memory-efficient.


In [ ]:
# ✅ Test Ex 2B.1
test_fasta = OUT_DIR / "headers_test.fasta"
with open(test_fasta, "w", encoding="utf-8") as f:
    f.write(">gene1 Homo sapiens TP53\nATGC\n>gene2\nGGCC\n>gene3 Mus musculus Trp53\nTTTT\n")

h_dict = parse_fasta_headers(test_fasta)
assert len(h_dict) == 3, f"Expected 3 headers, got {len(h_dict)}"
assert h_dict["gene1"] == "Homo sapiens TP53"
assert h_dict["gene2"] == ""
assert h_dict["gene3"] == "Mus musculus Trp53"
print("Ex 2B.1 passed ✔")


### Ex 2B.2 (★) — Calculate Sequence Lengths from FASTA

**Problem Statement:**  
Write a function `calculate_fasta_lengths(filepath: Path) -> dict[str, int]` that reads a multi-FASTA file (handling wrapped lines) and returns a dictionary mapping `{seq_id: sequence_length_in_bp}`.

- **Input:** `filepath: Path`.
- **Output:** `dict[str, int]`.


In [ ]:
def calculate_fasta_lengths(filepath: Path) -> dict[str, int]:
    lengths = {}
    current_id = None
    curr_len = 0

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            cleaned = line.strip()
            if not cleaned:
                continue
            if cleaned.startswith(">"):
                if current_id is not None:
                    lengths[current_id] = curr_len
                current_id = cleaned[1:].strip().split()[0]
                curr_len = 0
            else:
                curr_len += len(cleaned)

        if current_id is not None:
            lengths[current_id] = curr_len

    return lengths


**Explanation:**
- Rather than storing the entire sequence string in memory, we simply increment an integer counter `curr_len += len(cleaned)`.
- This calculates lengths in $O(1)$ memory per record, making it exceptionally fast even on multi-gigabyte chromosome FASTA files.


In [ ]:
# ✅ Test Ex 2B.2
lengths = calculate_fasta_lengths(DATA_DIR / "sequences" / "multi_sample.fasta")
assert len(lengths) == 15, f"Expected 15 records, got {len(lengths)}"
assert lengths["sample_01"] == 407, f"Expected 407 bp, got {lengths['sample_01']}"
assert lengths["sample_07"] == 105, f"Expected 105 bp, got {lengths['sample_07']}"
print("Ex 2B.2 passed ✔")


### Ex 2B.3 (★★) — Split Multi-FASTA by Length Threshold

**Problem Statement:**  
Write a function `split_fasta_by_length_threshold(in_path: Path, short_path: Path, long_path: Path, threshold: int = 250) -> tuple[int, int]` that:
1. Parses `in_path`.
2. Sequences with length $< \text{threshold}$ are written to `short_path`.
3. Sequences with length $\ge \text{threshold}$ are written to `long_path`.
4. Sequence lines in both output files MUST be wrapped at 60 characters.
5. Returns a tuple `(short_count, long_count)`.

- **Input:** `in_path: Path`, `short_path: Path`, `long_path: Path`, `threshold: int`.
- **Output:** `tuple[int, int]`.


In [ ]:
def split_fasta_by_length_threshold(in_path: Path, short_path: Path, long_path: Path, threshold: int = 250) -> tuple[int, int]:
    # Helper to parse records
    records = read_fasta_by_hand(in_path)
    
    short_count = 0
    long_count = 0
    
    with open(short_path, "w", encoding="utf-8") as f_short, open(long_path, "w", encoding="utf-8") as f_long:
        for seq_id, seq in records.items():
            target_f = f_short if len(seq) < threshold else f_long
            if len(seq) < threshold:
                short_count += 1
            else:
                long_count += 1
            
            target_f.write(f">{seq_id}\n")
            for i in range(0, len(seq), 60):
                target_f.write(seq[i : i + 60] + "\n")
                
    return short_count, long_count


**Explanation:**
- We reuse `read_fasta_by_hand(in_path)` to parse multi-line sequences reliably.
- Opening both output files simultaneously (`with open(...) as f_short, open(...) as f_long:`) allows writing directly to the appropriate destination in a single pass.
- Sequences are cleanly wrapped at 60 characters using `seq[i : i + 60]`.


In [ ]:
# ✅ Test Ex 2B.3
short_out = OUT_DIR / "short_seqs.fasta"
long_out = OUT_DIR / "long_seqs.fasta"

n_short, n_long = split_fasta_by_length_threshold(
    DATA_DIR / "sequences" / "multi_sample.fasta",
    short_out,
    long_out,
    threshold=250
)

assert n_short + n_long == 15, "Sum of counts must equal total records (15)"
assert n_short == 8, f"Expected 8 short records, got {n_short}"
assert n_long == 7, f"Expected 7 long records, got {n_long}"
assert short_out.exists() and long_out.exists()
print("Ex 2B.3 passed ✔")


### Ex 2B.4 (★★) — Generate Reverse Complement Multi-FASTA

**Problem Statement:**  
Write a function `generate_reverse_complement_fasta(in_path: Path, out_path: Path) -> int` that:
1. Reads all sequences from `in_path`.
2. Computes the reverse complement for each DNA sequence:
   - A $\leftrightarrow$ T, C $\leftrightarrow$ G, N $\leftrightarrow$ N (case-insensitive, preserving uppercase output).
   - Reverses the orientation (5′ $\rightarrow$ 3′).
3. Writes the reverse-complemented sequences to `out_path` wrapped at 60 bp.
4. Appends `_rc` to the sequence identifier in the header (e.g. `>sample_01_rc`).
5. Returns the number of records converted.

- **Input:** `in_path: Path`, `out_path: Path`.
- **Output:** `int` total records written.


In [ ]:
def generate_reverse_complement_fasta(in_path: Path, out_path: Path) -> int:
    trans_table = str.maketrans("ACGTNacgtn", "TGCANtgcan")
    records = read_fasta_by_hand(in_path)
    count = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for seq_id, seq in records.items():
            # Complement then reverse
            rev_comp = seq.translate(trans_table)[::-1].upper()
            f.write(f">{seq_id}_rc\n")
            for i in range(0, len(rev_comp), 60):
                f.write(rev_comp[i : i + 60] + "\n")
            count += 1

    return count


**Explanation:**
- `str.maketrans()` and `.translate()` perform character substitution at C-level speed.
- `[::-1]` performs sequence reversal.
- Suffixing `_rc` ensures header identifiers remain unique and self-documenting.


In [ ]:
# ✅ Test Ex 2B.4
rc_out = OUT_DIR / "multi_sample_rc.fasta"
count = generate_reverse_complement_fasta(DATA_DIR / "sequences" / "multi_sample.fasta", rc_out)

assert count == 15, f"Expected 15 records, got {count}"
rc_records = read_fasta_by_hand(rc_out)
assert "sample_01_rc" in rc_records
# Check reverse complement of sample_01
orig_sample_01 = parsed_dict["sample_01"]
rc_sample_01 = rc_records["sample_01_rc"]
assert len(rc_sample_01) == len(orig_sample_01)
# 5' end of RC should complement 3' end of original
assert rc_sample_01[:5] == orig_sample_01[-5:].translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
print("Ex 2B.4 passed ✔")


### Ex 2B.5 (★★) — Filter FASTQ Reads by Mean Phred Quality

**Problem Statement:**  
Write a function `filter_fastq_by_mean_quality(fastq_path: Path, min_q: float) -> list[tuple[str, float]]` that:
1. Streams `fastq_path` in 4-line blocks.
2. Extracts read ID from line 1 and quality string from line 4.
3. Converts the quality string to Phred+33 integers ($Q = \text{ord}(c) - 33$).
4. Computes the arithmetic mean Phred score for the read.
5. If the mean Phred score is $\ge \text{min\_q}$, includes `(read_id, round(mean_q, 2))` in the output list.
6. Returns the list of qualifying `(read_id, mean_score)` tuples.

- **Input:** `fastq_path: Path`, `min_q: float`.
- **Output:** `list[tuple[str, float]]`.


In [ ]:
def filter_fastq_by_mean_quality(fastq_path: Path, min_q: float) -> list[tuple[str, float]]:
    passed_reads = []
    with open(fastq_path, "r", encoding="utf-8") as f:
        while True:
            l1 = f.readline()
            if not l1:
                break
            l2 = f.readline()
            l3 = f.readline()
            l4 = f.readline()
            
            read_id = l1.strip()[1:].split()[0]
            qual_str = l4.strip()
            
            scores = [ord(c) - 33 for c in qual_str]
            mean_q = sum(scores) / len(scores) if scores else 0.0
            
            if mean_q >= min_q:
                passed_reads.append((read_id, round(mean_q, 2)))
                
    return passed_reads


**Explanation:**
- Streaming with 4 consecutive `f.readline()` calls ensures constant $O(1)$ memory usage.
- Converting ASCII to Phred with `ord(c) - 33` matches the Phred+33 standard.
- Rounding to 2 decimal places ensures clean, consistent floating-point comparisons.


In [ ]:
# ✅ Test Ex 2B.5
fastq_path = DATA_DIR / "sequences" / "reads_small.fastq"
high_q_reads = filter_fastq_by_mean_quality(fastq_path, min_q=30.0)

assert len(high_q_reads) > 0, "Should find high-quality reads"
assert all(score >= 30.0 for _, score in high_q_reads)
assert high_q_reads[0][0].startswith("READ_")
print(f"Passed reads (Q>=30): {len(high_q_reads)}")
print("Ex 2B.5 passed ✔")


### Ex 2B.6 (★★★) — Batch Chunking of a Multi-FASTA File

**Problem Statement:**  
In high-performance bioinformatics pipelines, large multi-sequence files are frequently chunked into batches of $N$ records each to parallelize jobs across compute nodes.

Write a function `batch_chunk_multi_fasta(in_path: Path, out_dir: Path, records_per_file: int = 5) -> list[Path]` that:
1. Parses all records from `in_path`.
2. Partitions records into consecutive batches of size `records_per_file`.
3. Writes each batch to a separate file named `chunk_001.fasta`, `chunk_002.fasta`, etc., inside `out_dir`.
4. Wraps sequences at 60 characters in every output chunk.
5. Returns a list of `Path` objects pointing to the created chunk files.

- **Input:** `in_path: Path`, `out_dir: Path`, `records_per_file: int`.
- **Output:** `list[Path]`.


In [ ]:
def batch_chunk_multi_fasta(in_path: Path, out_dir: Path, records_per_file: int = 5) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    records = read_fasta_by_hand(in_path)
    record_items = list(records.items())
    
    chunk_paths = []
    chunk_idx = 1
    
    for i in range(0, len(record_items), records_per_file):
        batch = record_items[i : i + records_per_file]
        chunk_file = out_dir / f"chunk_{chunk_idx:03d}.fasta"
        
        with open(chunk_file, "w", encoding="utf-8") as f:
            for seq_id, seq in batch:
                f.write(f">{seq_id}\n")
                for j in range(0, len(seq), 60):
                    f.write(seq[j : j + 60] + "\n")
                    
        chunk_paths.append(chunk_file)
        chunk_idx += 1
        
    return chunk_paths


**Explanation:**
- `range(0, len(record_items), records_per_file)` chunks the list into consecutive slices.
- `f"chunk_{chunk_idx:03d}.fasta"` uses zero-padded 3-digit formatting (`001`, `002`), ensuring natural alphabetical file ordering in shell directories.
- All created `Path` objects are collected and returned for seamless downstream workflow chaining.


In [ ]:
# ✅ Test Ex 2B.6
chunks_dir = OUT_DIR / "fasta_chunks"
created_chunks = batch_chunk_multi_fasta(
    DATA_DIR / "sequences" / "multi_sample.fasta",
    chunks_dir,
    records_per_file=5
)

assert len(created_chunks) == 3, f"15 records / 5 per file = 3 chunks, got {len(created_chunks)}"
assert created_chunks[0].name == "chunk_001.fasta"
assert created_chunks[2].name == "chunk_003.fasta"

# Verify record counts in first chunk
c1_records = read_fasta_by_hand(created_chunks[0])
assert len(c1_records) == 5, f"Expected 5 records in chunk 1, got {len(c1_records)}"
print("Ex 2B.6 passed ✔")


---
## 6. Summary & Cheat Sheet

| Task | Syntax / Formula | Key Considerations |
|---|---|---|
| FASTA Header Check | `line.startswith(">")` | Strip leading `>`: `line[1:].strip()` |
| Extract ID & Desc | `line[1:].strip().split(maxsplit=1)` | ID is first token; rest is description |
| Multi-line Sequence | Chunks in list: `chunks.append(line.strip())` | Join with `"".join(chunks)` |
| Post-Loop Save | `if current_id is not None: records[current_id] = ...` | **Must save the last record in file!** |
| Sequence Line Wrap | `for i in range(0, len(seq), 60): seq[i:i+60]` | Prevents buffer overflows in legacy tools |
| Reverse Complement | `seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]` | Complement first, then reverse slice |
| FASTQ Line 4 Decode | `scores = [ord(c) - 33 for c in qual_str]` | Phred+33 ASCII offset mapping |
| Phred Error Rate | $P = 10^{-Q / 10}$ | $Q30 = 99.9\%$ accuracy benchmark |

---


## 7. Further Reading & Resources

- [NCBI: FASTA Format Description](https://www.ncbi.nlm.nih.gov/genbank/fastaformat/)
- [Cock et al. (2010): The Sanger FASTQ file format for sequences with quality scores](https://academic.oup.com/nar/article/38/6/1767/3112533)
- [Illumina: Quality Scores for Next-Generation Sequencing](https://www.illumina.com/documents/products/technotes/technote_Q-Scores.pdf)
- [Biopython SeqIO Tutorial](https://biopython.org/DIST/docs/tutorial/Tutorial.html#sec14)
